# Análisis Inteligente del Consumo Energético

El objetivo de este proyecto es desarrollar un modelo de Machine Learning capaz de clasificar el nivel de eficiencia energética de una vivienda o pequeño establecimiento a partir de diferentes características relacionadas con el consumo eléctrico.

Se generará un dataset sintético que represente distintos perfiles de consumo.

Posteriormente se realizará un análisis exploratorio de los datos (EDA), el entrenamiento de modelos supervisados y la evaluación de su desempeño para seleccionar el modelo más adecuado.

# Variables del Dataset

Para representar el comportamiento energético de una vivienda se seleccionaron las siguientes variables.

1. Fecha

      Tipo: Fecha (YYYY-MM)

      - Representa el período al que corresponde el consumo energético registrado.

      Permite realizar análisis temporales como:

      - Evolución del consumo.

      - Comparación entre meses.

      - Tendencias de eficiencia.

      - Variaciones estacionales.

2. Consumo mensual (kWh)

      Tipo: Numérica

      Representa la cantidad de energía consumida durante un mes.

      Es la variable principal del problema y será utilizada para:

      - Estimar el costo mensual.

      - Analizar patrones de consumo.

      - Detectar consumos elevados.
      
3. Uso en horario pico

      Tipo: Booleana

      Indica si la mayor parte del consumo ocurre durante los horarios de mayor demanda eléctrica.

      Su análisis permite:

      - Detectar hábitos poco eficientes.

      - Generar recomendaciones para redistribuir el consumo.

4. Horas de alto consumo

      Tipo: Numérica

      Cantidad aproximada de horas diarias durante las cuales el consumo eléctrico es elevado.

      Permite identificar:

      - Intensidad del uso energético.

      - Relación entre tiempo de uso y consumo mensual.

5. Cantidad de equipos eléctricos

      Tipo: Numérica

      Número de electrodomésticos o equipos eléctricos utilizados regularmente.

      - Esta variable permite analizar cómo influye la cantidad de dispositivos sobre el consumo energético.

6. Cantidad de personas

      Tipo: Numérica

      - Cantidad de personas que habitan el inmueble.

      Se incorpora debido a que el consumo suele aumentar con el número de habitantes.

7. Tipo de inmueble

      Tipo: Categórica

      Puede tomar los valores:

      - Casa

      - Departamento

      - Comercio

      Permite analizar diferencias de consumo entre distintos tipos de establecimientos.

8. Categoría de eficiencia energética (Target)

      Tipo: Categórica

      Representa la clasificación que el modelo deberá aprender.

      Se utilizarán tres categorías:

      - Eficiente

      - Moderado
      
      - Ineficiente

      Las etiquetas serán generadas mediante un conjunto de reglas que consideran el consumo energético y otros factores relacionados con los hábitos de uso.

# Generación del Dataset

Se desarrolló un generador de datos sintéticos.

El objetivo es representar distintos perfiles de consumo energético manteniendo relaciones coherentes entre las variables.

Se generarán 1000 registros correspondientes a distintos inmuebles con características variadas.

Posteriormente estas observaciones serán utilizadas para entrenar y evaluar modelos supervisados de clasificación.

In [ ]:
import random
import pandas as pd
from datetime import datetime

NUM_REGISTROS = 1000

TIPOS_INMUEBLE = [
    "Casa",
    "Departamento",
    "Comercio"
]
def generar_fecha():

    año = random.randint(2023, 2025)
    mes = random.randint(1, 12)

    fecha = f"{año}-{mes:02d}"

    return fecha, mes


def generar_tipo():
    return random.choices(
        TIPOS_INMUEBLE,
        weights=[50, 35, 15]
    )[0]


def generar_personas(tipo):

    if tipo == "Casa":
        return random.randint(2, 6)

    elif tipo == "Departamento":
        return random.randint(1, 4)

    return random.randint(2, 8)


def generar_equipos(personas):
    return random.randint(personas + 2, personas + 10)


def generar_horas():
    return random.randint(2, 12)


def generar_horario_pico():
    return random.random() < 0.65


def calcular_consumo(personas, equipos, horas, horario_pico, mes):

    consumo = 80

    consumo += personas * random.randint(20, 35)

    consumo += equipos * random.randint(6, 10)

    consumo += horas * random.randint(8, 15)

    if horario_pico:
        consumo += random.randint(20, 50)


    # Estacionalidad
    if mes in [12, 1, 2]:           # Verano
        consumo += random.randint(40,80)

    elif mes in [6, 7, 8]:          # Invierno
        consumo += random.randint(30,70)

    consumo += random.randint(-15,15)

    return max(consumo,100)

    consumo += random.randint(-15, 15)

    return max(consumo, 100)

#INdice para calcular categoria

def calcular_indice(consumo,
                    horario_pico,
                    horas,
                    equipos,
                    personas):

    indice = 100

    if consumo > 600:
        indice -= 40
    elif consumo > 450:
        indice -= 25
    elif consumo > 300:
        indice -= 10

    if horario_pico:
        indice -= 15

    if horas >= 10:
        indice -= 15
    elif horas >= 7:
        indice -= 8


    if equipos >= 15:
        indice -= 15
    elif equipos >= 10:
        indice -= 8


    if personas >= 5:
        indice -= 5

    indice += random.randint(-8, 8)

    indice = max(0, min(indice, 100))

    return indice

def clasificar(indice):

    if indice >= 75:
        return "Eficiente"

    elif indice >= 45:
        return "Moderado"

    return "Ineficiente"


def generar_registro():

    fecha, mes = generar_fecha()

    tipo = generar_tipo()

    personas = generar_personas(tipo)

    equipos = generar_equipos(personas)

    horas = generar_horas()

    horario_pico = generar_horario_pico()

    consumo = calcular_consumo(
        personas,
        equipos,
        horas,
        horario_pico,
        mes
    )

    indice = calcular_indice(
        consumo,
        horario_pico,
        horas,
        equipos,
        personas
    )

    categoria = clasificar(indice)

    return {
        "fecha": fecha,
        "consumo_kwh": consumo,
        "uso_horario_pico": horario_pico,
        "horas_alto_consumo": horas,
        "cantidad_equipos": equipos,
        "cantidad_personas": personas,
        "tipo_inmueble": tipo,
        "categoria": categoria
    }
dataset = []

for _ in range(NUM_REGISTROS):
    dataset.append(generar_registro())

df = pd.DataFrame(dataset)

df.to_csv(
    "consumo_energetico.csv",
    index=False,
    encoding="utf-8"
)

print(df.head())

print("\nDistribución de categorías:")
print(df["categoria"].value_counts())

     fecha  consumo_kwh  uso_horario_pico  horas_alto_consumo  \
0  2024-02          449              True                   9   
1  2023-04          411              True                   8   
2  2023-10          393              True                   2   
3  2025-12          496              True                   8   
4  2025-05          217              True                   5   

   cantidad_equipos  cantidad_personas tipo_inmueble    categoria  
0                13                  3  Departamento     Moderado  
1                11                  4  Departamento     Moderado  
2                14                  8      Comercio     Moderado  
3                13                  6      Comercio  Ineficiente  
4                 5                  1  Departamento    Eficiente  

Distribución de categorías:
categoria
Moderado       535
Eficiente      328
Ineficiente    137
Name: count, dtype: int64


#Conociendo el Dataset

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("consumo_energetico.csv")

In [ ]:
print("Primeros registros")
df.head()

Primeros registros


,fecha,consumo_kwh,uso_horario_pico,horas_alto_consumo,cantidad_equipos,cantidad_personas,tipo_inmueble,categoria
0,2024-02,449,True,9,13,3,Departamento,Moderado
1,2023-04,411,True,8,11,4,Departamento,Moderado
2,2023-10,393,True,2,14,8,Comercio,Moderado
3,2025-12,496,True,8,13,6,Comercio,Ineficiente
4,2025-05,217,True,5,5,1,Departamento,Eficiente


In [ ]:
print("Últimos registros")
df.tail()

Últimos registros


,fecha,consumo_kwh,uso_horario_pico,horas_alto_consumo,cantidad_equipos,cantidad_personas,tipo_inmueble,categoria
995,2025-07,630,True,11,11,8,Comercio,Ineficiente
996,2024-12,534,False,7,13,8,Comercio,Moderado
997,2024-11,413,True,12,7,2,Departamento,Moderado
998,2023-03,441,True,6,14,4,Departamento,Moderado
999,2025-12,371,False,4,12,4,Casa,Eficiente


In [ ]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Filas: 1000
Columnas: 8


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   fecha               1000 non-null   object
 1   consumo_kwh         1000 non-null   int64 
 2   uso_horario_pico    1000 non-null   bool  
 3   horas_alto_consumo  1000 non-null   int64 
 4   cantidad_equipos    1000 non-null   int64 
 5   cantidad_personas   1000 non-null   int64 
 6   tipo_inmueble       1000 non-null   object
 7   categoria           1000 non-null   object
dtypes: bool(1), int64(4), object(3)
memory usage: 55.8+ KB


In [ ]:
df.describe()

,consumo_kwh,horas_alto_consumo,cantidad_equipos,cantidad_personas
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,385.355000,7.095000,9.586000,3.540000
std,84.467092,3.148951,3.017567,1.644112
min,171.000000,2.000000,3.000000,1.000000
25%,323.000000,4.000000,7.000000,2.000000
50%,383.000000,7.000000,10.000000,3.000000
75%,442.000000,10.000000,12.000000,5.000000
max,646.000000,12.000000,18.000000,8.000000


In [ ]:
print("Consumo promedio")

print(df["consumo_kwh"].mean())



Consumo promedio
385.355


In [ ]:
print("Consumo mínimo")
print(df["consumo_kwh"].min())

print("Consumo máximo")
print(df["consumo_kwh"].max())

Consumo mínimo
171
Consumo máximo
646


In [ ]:
df["categoria"].value_counts()

,count
categoria,
Moderado,535
Eficiente,328
Ineficiente,137


In [ ]:
round(
    df["categoria"].value_counts(normalize=True)*100,
    2
)

,proportion
categoria,
Moderado,53.5
Eficiente,32.8
Ineficiente,13.7


In [ ]:
df["tipo_inmueble"].value_counts()

,count
tipo_inmueble,
Casa,477
Departamento,378
Comercio,145


In [ ]:
round(
    df["tipo_inmueble"].value_counts(normalize=True)*100,
    2
)

,proportion
tipo_inmueble,
Casa,47.7
Departamento,37.8
Comercio,14.5


In [ ]:
df["uso_horario_pico"].value_counts()

,count
uso_horario_pico,
True,675
False,325


In [ ]:
round(
    df["uso_horario_pico"].value_counts(normalize=True)*100,
    2
)

,proportion
uso_horario_pico,
True,67.5
False,32.5


In [ ]:
df.groupby("tipo_inmueble")["consumo_kwh"].mean().sort_values(ascending=False)

,consumo_kwh
tipo_inmueble,
Comercio,424.255172
Casa,404.345912
Departamento,346.468254


In [ ]:
df.groupby("categoria")["consumo_kwh"].mean()

,consumo_kwh
categoria,
Eficiente,314.618902
Ineficiente,508.744526
Moderado,397.125234


#Modelo Inicial

### Selección del modelo

Para este proyecto se seleccionó **Random Forest** como algoritmo de clasificación debido a sus características y ventajas para este tipo de problema.

Las principales razones son:

- Trabaja muy bien con variables numéricas y categóricas.
- Es capaz de aprender relaciones no lineales entre las variables.
- Es menos propenso al sobreajuste que un único Árbol de Decisión, ya que combina las predicciones de múltiples árboles.
- Requiere poco preprocesamiento de los datos, ya que no necesita normalización o estandarización.
- Permite obtener la importancia de cada variable, facilitando la interpretación del modelo y el análisis de qué factores influyen más en la eficiencia energética.



In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"])

df["mes"] = df["fecha"].dt.month

In [ ]:
df = df.drop(columns=["fecha"])

Se define **X** como el conjunto de variables predictoras (features), es decir, todas aquellas características que describen el perfil de consumo energético y que serán utilizadas por el modelo para realizar la clasificación.

La variable **y** representa la variable objetivo (*target*), correspondiente a la categoría de eficiencia energética que el modelo deberá aprender a predecir.

In [ ]:
X = df.drop(columns=["categoria"])

y = df["categoria"]

In [ ]:
print(X.head())
print(y.head())

   consumo_kwh  uso_horario_pico  horas_alto_consumo  cantidad_equipos  \
0          449              True                   9                13   
1          411              True                   8                11   
2          393              True                   2                14   
3          496              True                   8                13   
4          217              True                   5                 5   

   cantidad_personas tipo_inmueble  mes  
0                  3  Departamento    2  
1                  4  Departamento    4  
2                  8      Comercio   10  
3                  6      Comercio   12  
4                  1  Departamento    5  
0       Moderado
1       Moderado
2       Moderado
3    Ineficiente
4      Eficiente
Name: categoria, dtype: object


Se utiliza la función `train_test_split` para dividir el conjunto de datos en un subconjunto de entrenamiento y otro de prueba.

El conjunto de datos se divide en:

- **80 %** para entrenamiento del modelo.
- **20 %** para evaluación del desempeño.

Además:

- **random_state = 42** garantiza que la división sea reproducible en futuras ejecuciones.
- **stratify = y** mantiene la misma proporción de cada categoría (Eficiente, Moderado e Ineficiente) tanto en el conjunto de entrenamiento como en el de prueba, evitando sesgos durante la evaluación.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

El modelo de Machine Learning requiere que todas las variables de entrada sean numéricas. Sin embargo, la variable **tipo_inmueble** contiene valores categóricos ("Casa", "Departamento" y "Comercio"), por lo que debe transformarse antes del entrenamiento.

Se utiliza un **ColumnTransformer** para aplicar transformaciones únicamente a las columnas que lo requieren.

En este caso:

- La columna **tipo_inmueble** se transforma mediante **One-Hot Encoding**, generando una columna binaria para cada categoría.
- El resto de las variables numéricas se mantienen sin modificaciones mediante la opción `remainder="passthrough"`.

El parámetro `handle_unknown="ignore"` permite que el modelo continúe funcionando incluso si durante la predicción aparece una categoría no observada durante el entrenamiento.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "tipo",
            OneHotEncoder(handle_unknown="ignore"),
            ["tipo_inmueble"]
        )
    ],
    remainder="passthrough"
)

Se utiliza un **Pipeline** para integrar el preprocesamiento y el modelo de clasificación en un único flujo de trabajo.

Esto garantiza que cualquier dato nuevo pase automáticamente por las mismas transformaciones antes de ser clasificado, evitando inconsistencias entre el entrenamiento y la predicción.

El Pipeline está compuesto por dos etapas:

1. **Preprocesamiento:** transformación de las variables categóricas mediante One-Hot Encoding.
2. **Clasificación:** entrenamiento del modelo Random Forest.

Se configuró el algoritmo con **200 árboles de decisión** (`n_estimators=200`), buscando mejorar la estabilidad de las predicciones mediante el voto conjunto de múltiples árboles.

El parámetro `random_state=42` garantiza que el entrenamiento sea reproducible, obteniendo los mismos resultados en ejecuciones posteriores.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

modelo = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

Finalmente, se entrena el modelo utilizando el conjunto de entrenamiento.

Durante esta etapa, Random Forest construye múltiples árboles de decisión a partir de diferentes subconjuntos de datos y variables. Cada árbol realiza una predicción de forma independiente y la clasificación final se obtiene mediante votación mayoritaria, lo que mejora la robustez y reduce el riesgo de sobreajuste.

In [ ]:
modelo.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('tipo',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['tipo_inmueble'])])),
                ('classifier',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

Una vez entrenado el modelo, se realizan predicciones utilizando el conjunto de prueba (X_test), compuesto por datos que el algoritmo nunca vio durante el entrenamiento.

Esta etapa permite evaluar la capacidad del modelo para generalizar y clasificar correctamente nuevos perfiles de consumo energético.

In [ ]:
predicciones = modelo.predict(X_test)

Además de indicar la categoría predicha, Random Forest puede estimar la probabilidad asociada a cada clase.

Esto permite conocer el nivel de confianza de la predicción realizada.

Por ejemplo, una salida como:

  [0.92, 0.03, 0.05]

  indica que el modelo considera un 92 % de probabilidad de que el registro pertenezca a la primera categoría.

In [ ]:
probabilidades = modelo.predict_proba(X_test)

La métrica Accuracy mide el porcentaje de predicciones correctas realizadas por el modelo.

Se calcula comparando las categorías predichas con las categorías reales del conjunto de prueba.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predicciones
)

print(accuracy)

0.865


El Classification Report proporciona una evaluación más completa del desempeño del modelo para cada una de las categorías de eficiencia energética.

Precision

- Indica qué tan confiables son las predicciones realizadas por el modelo.

Recall

- Mide la capacidad del modelo para identificar correctamente todos los registros pertenecientes a una determinada categoría.

F1-Score

- Es una medida que combina Precision y Recall en un único indicador.

Support

- Representa la cantidad de registros pertenecientes a cada categoría dentro del conjunto de prueba.

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predicciones
    )
)

              precision    recall  f1-score   support

   Eficiente       0.83      0.88      0.85        66
 Ineficiente       0.88      0.85      0.87        27
    Moderado       0.88      0.86      0.87       107

    accuracy                           0.86       200
   macro avg       0.87      0.86      0.86       200
weighted avg       0.87      0.86      0.87       200



La diagonal principal representa las predicciones correctas.

Todo lo que está fuera de la diagonal son errores de clasificación.

Eficiente(R)

Ineficiente(R)

Moderado(R)


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    predicciones
)

print(cm)

[[58  0  8]
 [ 0 23  4]
 [12  3 92]]


Random Forest permite estimar la importancia relativa de cada variable utilizada durante el entrenamiento.

Esta información indica cuáles fueron los factores que más contribuyeron a la toma de decisiones del modelo.

In [ ]:
import pandas as pd

importancias = modelo.named_steps["classifier"].feature_importances_

nombres = modelo.named_steps["preprocessor"].get_feature_names_out()

pd.DataFrame({
    "Variable": nombres,
    "Importancia": importancias
}).sort_values(
    by="Importancia",
    ascending=False
)

,Variable,Importancia
3,remainder__consumo_kwh,0.384836
5,remainder__horas_alto_consumo,0.167670
6,remainder__cantidad_equipos,0.138899
4,remainder__uso_horario_pico,0.119559
7,remainder__cantidad_personas,0.080964
8,remainder__mes,0.075041
2,tipo__tipo_inmueble_Departamento,0.013646
0,tipo__tipo_inmueble_Casa,0.011536
1,tipo__tipo_inmueble_Comercio,0.007848


In [ ]:
import joblib

joblib.dump(
    modelo,
    "modelo_consumo.pkl"
)

['modelo_consumo.pkl']

In [ ]:
datosClientes = pd.DataFrame([{
    "consumo_kwh": 420,
    "uso_horario_pico": True,
    "horas_alto_consumo": 8,
    "cantidad_equipos": 10,
    "cantidad_personas": 4,
    "tipo_inmueble": "Casa",
    "mes": 7
}])

In [ ]:
categoria = modelo.predict(datosClientes)[0]

probabilidades = modelo.predict_proba(datosClientes)[0]

indice = list(modelo.classes_).index(categoria)

probabilidad = probabilidades[indice]

print(categoria)
print(probabilidad)

Moderado
0.975


In [ ]:
nuevo = pd.DataFrame([{
    "consumo_kwh": 180,
    "uso_horario_pico": False,
    "horas_alto_consumo": 3,
    "cantidad_equipos": 5,
    "cantidad_personas": 2,
    "tipo_inmueble": "Departamento",
    "mes": 5
}])

print(modelo.predict(nuevo))
print(modelo.predict_proba(nuevo))

['Eficiente']
[[1. 0. 0.]]


In [ ]:
nuevo = pd.DataFrame([{
    "consumo_kwh": 620,
    "uso_horario_pico": True,
    "horas_alto_consumo": 11,
    "cantidad_equipos": 16,
    "cantidad_personas": 6,
    "tipo_inmueble": "Casa",
    "mes": 7
}])

print(modelo.predict(nuevo))
print(modelo.predict_proba(nuevo))

['Ineficiente']
[[0.   0.98 0.02]]


In [ ]:
nuevo = pd.DataFrame([{
    "consumo_kwh": 340,
    "uso_horario_pico": False,
    "horas_alto_consumo": 5,
    "cantidad_equipos": 8,
    "cantidad_personas": 3,
    "tipo_inmueble": "Casa",
    "mes": 10
}])

print(modelo.predict(nuevo))
print(modelo.predict_proba(nuevo))

['Eficiente']
[[0.96 0.   0.04]]


In [ ]:
import os

tam = os.path.getsize("modelo_consumo.pkl")

print(f"{tam / 1024:.2f} KB")

4935.37 KB
